# Search Expert

## Try it in 5 minutes

<a target="_blank" href="https://colab.research.google.com/github/sarthakrastogi/search-expert/blob/main/examples/search_expert_colab.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## What is Search Expert?

Search Expert is a lightweight Python library that uses a custom fine-tuned Small Language Model to convert natural-language search queries into clean, structured queries.

It can be used in search engines in ecommerce search, travel search, and virtually any use case where filtering based on user query understanding is needed.

    "Non-stop business class from JFK to Tokyo under $3,000"
                  ↓
    {
      "domain":       "flights",
      "origin":       "JFK",
      "destination":  "Tokyo",
      "cabin_class":  "business",
      "stops":        "lte:0",
      "price":        "lt:3000"
    }



> IMPORTANT: For best speed, enable a GPU instance on Colab via Edit > Notebook Settings > t4 GPU



------------------------------------------------------------------------

## Install

In [1]:
%%capture
!pip install search-expert

## Basic usage

In [2]:

from search_expert import SearchExpert, ModelFormat, ParseResult

expert = SearchExpert()  # loads the JSON adapter by default

In [3]:

result = expert.parse("noise cancelling headphones any colour but red or green, under $200")
print(f"OUTPUT QUERY: {result.fields}")

#{
#      'domain':   'ecommerce',
#      'product':  'headphones',
#      'feature':  'noise cancelling',
#      'color':    ['ne:red', 'ne:green'],
#      'price':    'lt:200'
#    }


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.6: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

OUTPUT QUERY: {'domain': 'ecommerce', 'product': 'headphones', 'feature': 'noise cancelling', 'color': ['ne:red', 'ne:green'], 'price': 'lt:200'}


------------------------------------------------------------------------

## Choosing an output format

In [4]:
expert_yaml = SearchExpert(fmt=ModelFormat.YAML)

result = expert_yaml.parse("Senior ML engineer jobs in London paying over £120k")
print(f"OUTPUT QUERY: {result.fields}")

==((====))==  Unsloth 2026.4.6: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

OUTPUT QUERY: {'city': 'London', 'domain': 'jobs', 'experience_level': 'Senior', 'industry': 'ML', 'salary': 'gt:120000'}



## Batch parsing

In [5]:

queries = [
    "7-seater hybrid SUV under $40,000 with AWD",
    "5-star hotel in Paris with free breakfast under $300/night",
    "Taylor Swift concerts in London in July",
]

results = expert.parse_batch(queries)

for q, r in zip(queries, results):
    print(f"Q: {q}")
    print(f"   → {r.fields}\n")

Q: 7-seater hybrid SUV under $40,000 with AWD
   → {'domain': 'cars', 'body_type': 'SUV', 'fuel_type': 'hybrid', 'price': 'lt:40000', 'feature': 'AWD'}

Q: 5-star hotel in Paris with free breakfast under $300/night
   → {'domain': 'hotels', 'stars': '5', 'city': 'Paris', 'price': 'lt:300'}

Q: Taylor Swift concerts in London in July
   → {'domain': 'events', 'artist': 'Taylor Swift', 'city': 'London', 'month': 'July'}



------------------------------------------------------------------------

## Operator-prefixed values

Numeric and categorical constraints are encoded with a consistent prefix
syntax so downstream code can filter results without any
natural-language ambiguity:

| Query phrase                             | Output value             |
|------------------------------------------|--------------------------|
| “under \$200”, “below \$200”             | `lt:200`                 |
| “up to \$200”, “max \$200”               | `lte:200`                |
| “over \$150k”, “above \$150k”            | `gt:150000`              |
| “at least \$150k”, “\$150k+”             | `gte:150000`             |
| “around \$200”, “\~\$200”                | `approx:200`             |
| “\$100–\$200”, “between \$100 and \$200” | `between:100:200`        |
| “any colour but red or green”            | `["ne:red", "ne:green"]` |

------------------------------------------------------------------------

## The dataset

The models were fine-tuned on a private dataset of \~1 million (query,
structured output) pairs spanning 10 search domains:

-   Real estate, E-commerce, Jobs, Flights, Hotels
-   Cars, Restaurants, Movies / Streaming, Healthcare, Courses, Events

Each row contains the natural-language query plus the ground-truth
output in five formats simultaneously: JSON, YAML, TOML, CSV key-value,
and XML. The dataset was split 90 / 5 / 5 into train, validation, and
test.

------------------------------------------------------------------------

## The models

Each adapter is a LoRA fine-tune of **Qwen3.5-0.8B** (loaded in 4-bit)
trained for 300 steps with:

-   LoRA rank 16, cosine schedule, AdamW 8-bit
-   Effective batch size 16 (batch 8 × grad accum 2)
-   One adapter per output format (JSON, YAML, …)

At inference the model runs in `<1 second per query` on a T4 GPU and
under `3 seconds` on CPU.

------------------------------------------------------------------------

## How training works (for contributors)

The training pipeline lives in `finetune.py` and `evaluate.py` (see the
GitHub repo). At a high level:

1.  **One adapter per format** — the base model is loaded fresh for each
    output format, so adapters don’t interfere with each other.
2.  **Prompt format** — ChatML-style (`<|im_start|>` / `<|im_end|>`)
    with a system prompt that names the target format and forbids
    hallucination.
3.  **LoRA** — only the attention and MLP projection weights are
    fine-tuned (rank 16), keeping the adapter small (\~40 MB).
4.  **Evaluation** — after training, `evaluate.py` measures key-field
    F1, value exact-match accuracy, parse rate, and per-query latency on
    the held-out test split.


------------------------------------------------------------------------

## Format leaderboard (reference)

After training on the full dataset:

| Rank | Format        | Key F1 | Value Acc | Parse % |
|------|---------------|--------|-----------|---------|
| 🥇   | JSON          | 0.913  | 0.874     | 98.2%   |
| 🥈   | YAML          | 0.901  | 0.861     | 97.6%   |
| 🥉   | TOML          | 0.887  | 0.843     | 96.1%   |
| 4\.  | XML           | 0.871  | 0.829     | 94.8%   |
| 5\.  | CSV key=value | 0.856  | 0.812     | 93.3%   |

JSON is the recommended format for most use cases. YAML is a good
alternative when human readability of model output matters.